# Avance 2 · SSC — Etapa 1: delimitación de datos
**Equipo A&C · Proyecto de Inteligencia de Negocios**

**Pregunta de negocio:** ¿En qué alcaldías y horarios de la CDMX se concentra la mayor frecuencia de hechos de tránsito y en cuáles se observa mayor severidad registrada (personas lesionadas y fallecidas)?

En este notebook delimitamos los **cinco archivos SSC** a registros de **2018 en adelante** y a las **16 alcaldías de la CDMX**. Conservamos los datos crudos sin modificaciones. No eliminamos duplicados en esta etapa; la revisión de duplicidad y de folios corresponde a la Etapa 2.

**Orden de ejecución:** ejecutamos las celdas de arriba hacia abajo mediante *Run → Run All Cells*. Este notebook es autónomo: no requiere `funciones_calidad.py`.

## 1. Bibliotecas y localización del ZIP
Buscamos `CARPETA_DATOS.zip` en la carpeta actual y sus carpetas superiores. Si no aparece, realizamos una búsqueda en nuestra carpeta personal. Si hay varias copias, indicamos manualmente cuál utilizar en `RUTA_ZIP_MANUAL`. La carpeta de resultados se crea **junto al ZIP** para evitar rutas inconsistentes.

In [1]:
from pathlib import Path
import zipfile
import io
import re
import json
import hashlib
import unicodedata
from datetime import datetime
import pandas as pd
from IPython.display import display

RUTA_ZIP_MANUAL = None  # Ejemplo: Path('/ruta/a/CARPETA_DATOS.zip')

if RUTA_ZIP_MANUAL is not None:
    ZIP = Path(RUTA_ZIP_MANUAL).expanduser().resolve()
else:
    candidatos = []
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        candidatos.extend(carpeta.glob('CARPETA_DATOS.zip'))
    if not candidatos:
        candidatos = list(Path.home().rglob('CARPETA_DATOS.zip'))
    candidatos = sorted(set(p.resolve() for p in candidatos))
    if not candidatos:
        raise FileNotFoundError('No encontramos CARPETA_DATOS.zip. Sube el ZIP a JupyterLab o configura RUTA_ZIP_MANUAL.')
    if len(candidatos) > 1:
        print('Se encontraron varias copias; utilizaremos la primera. Puedes elegir otra con RUTA_ZIP_MANUAL:')
        for p in candidatos: print(' -', p)
    ZIP = candidatos[0]

assert ZIP.is_file(), f'No existe el ZIP: {ZIP}'
DESTINO = ZIP.parent / 'resultados_SSC' / 'etapa1'
DESTINO.mkdir(parents=True, exist_ok=True)
print('ZIP original:', ZIP)
print('Salida Etapa 1:', DESTINO)
print('Pandas:', pd.__version__)

ZIP original: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/CARPETA_DATOS.zip
Salida Etapa 1: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados_SSC/etapa1
Pandas: 1.4.2


## 2. Inventario y contrato de lectura
Identificamos los cinco CSV originales dentro del ZIP. Leemos sus bytes directamente del archivo comprimido, sin extraerlos ni modificar su contenido. Registramos la codificación utilizada, dimensiones, columnas y huella SHA-256 para trazabilidad.

In [2]:
ARCHIVOS_ESPERADOS = [
    'HechosTransito_SSC.csv',
    'PersonasLyFporEdad_SSC.csv',
    'PersonasLyFporSexo_SSC.csv',
    'PersonasLyFporTipoPersona_SSC.csv',
    'VehiculosInvolucrados_SSC.csv',
]
with zipfile.ZipFile(ZIP, 'r') as z:
    contenido = [x for x in z.namelist() if x.lower().endswith('.csv')]
    rutas_ssc = {}
    for nombre in ARCHIVOS_ESPERADOS:
        encontrados = [x for x in contenido if Path(x).name.lower() == nombre.lower() and '/ssc/' in ('/' + x.lower())]
        if len(encontrados) != 1:
            raise ValueError(f'Esperábamos una ruta única para {nombre}; encontramos {encontrados}')
        rutas_ssc[nombre] = encontrados[0]
print('Archivos SSC localizados:', len(rutas_ssc))
for nombre, ruta in rutas_ssc.items(): print(nombre, '→', ruta)

def leer_csv_zip(z, ruta):
    crudo = z.read(ruta)
    for codificacion in ['utf-8-sig', 'utf-8', 'cp1252', 'latin1']:
        try:
            df = pd.read_csv(io.BytesIO(crudo), dtype=str, encoding=codificacion, low_memory=False)
            return df, {'ruta_zip': ruta, 'codificacion': codificacion, 'separador': ',',
                        'sha256': hashlib.sha256(crudo).hexdigest(),
                        'filas': len(df), 'columnas': len(df.columns),
                        'encabezados': list(df.columns)}
        except UnicodeDecodeError:
            continue
    raise ValueError(f'No se pudo leer {ruta}')

Archivos SSC localizados: 5
HechosTransito_SSC.csv → CARPETA_DATOS/SSC/HechosTransito_SSC.csv
PersonasLyFporEdad_SSC.csv → CARPETA_DATOS/SSC/PersonasLyFporEdad_SSC.csv
PersonasLyFporSexo_SSC.csv → CARPETA_DATOS/SSC/PersonasLyFporSexo_SSC.csv
PersonasLyFporTipoPersona_SSC.csv → CARPETA_DATOS/SSC/PersonasLyFporTipoPersona_SSC.csv
VehiculosInvolucrados_SSC.csv → CARPETA_DATOS/SSC/VehiculosInvolucrados_SSC.csv


## 3. Criterios de delimitación
**Temporal:** interpretamos `fecha_evento` como `AAAA-MM-DD` (`%Y-%m-%d`) y conservamos fechas desde el 1 de enero de 2018.

**Geográfico:** comparamos el campo `alcaldia` con las 16 alcaldías, después de normalizar espacios, mayúsculas y acentos. Documentamos las equivalencias `CUAJIMALPA` → `CUAJIMALPA DE MORELOS`, `MAGDALENA CONTRERAS` → `LA MAGDALENA CONTRERAS` y `GUSTAVO A. MADERO` → `GUSTAVO A MADERO`.

No asignamos una alcaldía por inferencia: por ejemplo, `AV INSURGENTES` es una vialidad y requiere revisión.

In [3]:
ALCALDIAS = {
    'ALVARO OBREGON', 'AZCAPOTZALCO', 'BENITO JUAREZ', 'COYOACAN',
    'CUAJIMALPA DE MORELOS', 'CUAUHTEMOC', 'GUSTAVO A MADERO',
    'IZTACALCO', 'IZTAPALAPA', 'LA MAGDALENA CONTRERAS',
    'MIGUEL HIDALGO', 'MILPA ALTA', 'TLAHUAC', 'TLALPAN',
    'VENUSTIANO CARRANZA', 'XOCHIMILCO'
}
EQUIVALENCIAS = {
    'CUAJIMALPA': 'CUAJIMALPA DE MORELOS',
    'MAGDALENA CONTRERAS': 'LA MAGDALENA CONTRERAS',
    'GUSTAVO A. MADERO': 'GUSTAVO A MADERO'
}

def normalizar(valor):
    if pd.isna(valor): return ''
    texto = unicodedata.normalize('NFKD', str(valor).strip().upper())
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    return ' '.join(texto.split())

def normalizar_alcaldia(valor):
    nombre = normalizar(valor)
    return EQUIVALENCIAS.get(nombre, nombre)

assert len(ALCALDIAS) == 16
print('Alcaldías válidas:', len(ALCALDIAS))
print('Equivalencias documentadas:', EQUIVALENCIAS)

Alcaldías válidas: 16
Equivalencias documentadas: {'CUAJIMALPA': 'CUAJIMALPA DE MORELOS', 'MAGDALENA CONTRERAS': 'LA MAGDALENA CONTRERAS', 'GUSTAVO A. MADERO': 'GUSTAVO A MADERO'}


## 4. Perfil inicial de datos crudos
Antes de filtrar, registramos **todas las columnas** de cada archivo, su tipo aparente, número de valores faltantes, cantidad de valores distintos y ejemplos. Las tablas tienen distintas unidades de observación, por lo que no sumamos sus filas como si fueran hechos de tránsito únicos.

In [4]:
def perfil_columnas(df, archivo):
    registros = []
    for columna in df.columns:
        serie = df[columna]
        ejemplos = serie.dropna().astype(str).drop_duplicates().head(3).tolist()
        registros.append({
            'archivo': archivo, 'columna': columna, 'tipo_aparente': str(serie.dtype),
            'nulos': int(serie.isna().sum()),
            'porcentaje_nulos': round(100 * serie.isna().mean(), 2) if len(df) else 0,
            'valores_distintos': int(serie.nunique(dropna=True)),
            'ejemplos': ' | '.join(ejemplos)
        })
    return pd.DataFrame(registros)

perfiles = []
contratos = []
with zipfile.ZipFile(ZIP, 'r') as z:
    for nombre, ruta in rutas_ssc.items():
        datos, contrato = leer_csv_zip(z, ruta)
        contrato['archivo'] = nombre
        contratos.append(contrato)
        perfiles.append(perfil_columnas(datos, nombre))
        print(f'{nombre}: {len(datos):,} filas, {len(datos.columns)} columnas')
        display(datos.head(3))

perfil_crudo = pd.concat(perfiles, ignore_index=True)
perfil_crudo.to_csv(DESTINO / 'perfil_crudo_columnas_SSC.csv', index=False, encoding='utf-8-sig')
with open(DESTINO / 'contratos_lectura_SSC.json', 'w', encoding='utf-8') as f:
    json.dump(contratos, f, ensure_ascii=False, indent=2)
print('Guardamos perfil inicial y contratos de lectura.')

HechosTransito_SSC.csv: 134,079 filas, 26 columnas


,fecha_evento,hora_evento,tipo_evento,fecha_captura,folio,latitud,longitud,punto_1,punto_2,colonia,...,clasificacion_de_la_vialidad,sentido_de_circulacion,dia,prioridad,origen,unidad_medica_de_apoyo,matricula_unidad_medica,trasladado_lesionados,personas_fallecidas,personas_lesionadas
0,2020-04-06,12:50:00,CHOQUE,2020-04-17,BJ/200406/03499,19.368116,-99.142903,EJE 7 SUR,ANTILLAS,PORTALES NTE,...,EJE VIAL,P-O,Lunes,BAJA,RADIO,PC,NaN,NO,0,1
1,2020-04-06,18:31:00,CHOQUE,2020-04-17,C5/200406/05748,19.301142,-99.115521,CALZ DEL HUESO,RANCHO COLORADO,COAPA STA CECILIA,...,VIA PRIMARIA,O-P,Lunes,BAJA,911 CDMX,ERUM,NaN,NO,0,1
2,2020-04-06,18:39:00,CHOQUE,2020-04-17,C5/200406/05802,19.476843,-99.092207,EJE 5 NTE,AV GRAN CANAL DEL DESAGUE,JOSE MA MORELOS Y PAVON,...,EJE VIAL,P-O,Lunes,BAJA,911 CDMX,PARTICULAR,NaN,NO,0,1


PersonasLyFporEdad_SSC.csv: 158,077 filas, 23 columnas


,fecha_evento,hora_evento,tipo_evento,fecha_captura,folio,latitud,longitud,punto_1,punto_2,colonia,...,unidad_a_cargo,tipo_de_interseccion,interseccion_semaforizada,clasificacion_de_la_vialidad,sentido_de_circulacion,dia,prioridad,origen,condicion_persona,edad
0,2020-04-06,12:50:00,CHOQUE,2020-04-17,BJ/200406/03499,19.368116,-99.142903,EJE 7 SUR,ANTILLAS,PORTALES NTE,...,MX164B4,CRUZ,NO,EJE VIAL,P-O,Lunes,BAJA,RADIO,lesionada,17
1,2020-04-06,18:31:00,CHOQUE,2020-04-17,C5/200406/05748,19.301142,-99.115521,CALZ DEL HUESO,RANCHO COLORADO,COAPA STA CECILIA,...,MX337S2,CRUZ,NO,VIA PRIMARIA,O-P,Lunes,BAJA,911 CDMX,lesionada,34
2,2020-04-06,18:39:00,CHOQUE,2020-04-17,C5/200406/05802,19.476843,-99.092207,EJE 5 NTE,AV GRAN CANAL DEL DESAGUE,JOSE MA MORELOS Y PAVON,...,MX329N2,T,SI,EJE VIAL,P-O,Lunes,BAJA,911 CDMX,lesionada,30


PersonasLyFporSexo_SSC.csv: 144,044 filas, 24 columnas


...

PersonasLyFporTipoPersona_SSC.csv: 107,795 filas, 24 columnas


,fecha_evento,hora_evento,tipo_evento,fecha_captura,folio,latitud,longitud,punto_1,punto_2,colonia,...,tipo_de_interseccion,interseccion_semaforizada,clasificacion_de_la_vialidad,sentido_de_circulacion,dia,prioridad,origen,tipo_persona,condicion_persona,total
0,2020-04-06,12:50:00,CHOQUE,2020-04-17,BJ/200406/03499,19.368116,-99.142903,EJE 7 SUR,ANTILLAS,PORTALES NTE,...,CRUZ,NO,EJE VIAL,P-O,Lunes,BAJA,RADIO,Ciclista,lesionada,1
1,2020-04-06,18:31:00,CHOQUE,2020-04-17,C5/200406/05748,19.301142,-99.115521,CALZ DEL HUESO,RANCHO COLORADO,COAPA STA CECILIA,...,CRUZ,NO,VIA PRIMARIA,O-P,Lunes,BAJA,911 CDMX,Motociclista,lesionada,1
2,2020-04-06,18:39:00,CHOQUE,2020-04-17,C5/200406/05802,19.476843,-99.092207,EJE 5 NTE,AV GRAN CANAL DEL DESAGUE,JOSE MA MORELOS Y PAVON,...,T,SI,EJE VIAL,P-O,Lunes,BAJA,911 CDMX,Motociclista,lesionada,1


VehiculosInvolucrados_SSC.csv: 215,079 filas, 26 columnas


,fecha_evento,hora_evento,tipo_evento,fecha_captura,folio,latitud,longitud,punto_1,punto_2,colonia,...,clasificacion_de_la_vialidad,sentido_de_circulacion,dia,prioridad,origen,unidad_medica_de_apoyo,matricula_unidad_medica,trasladado_lesionados,no_vehiculo,tipo_vehiculo
0,2018-01-01,23:16:00,ATROPELLADO,2018-03-01,316753,19.437784,-99.140321,EJE CENTRAL,REP DE CUBA,CENTRO,...,NaN,NaN,Lunes,BAJA,NaN,CRUZ ROJA,NaN,NO,Vehículo 1,SIN DATO
1,2018-01-01,06:11:00,CHOQUE,2018-03-01,328860,19.496948,-99.073225,TIXTLA,SAN JUAN DE LOS LAGOS,SAN FELIPE DE JESUS,...,NaN,NaN,Lunes,MEDIA,NaN,ERUM,NaN,SI,Vehículo 1,AUTOMOVIL
2,2018-01-01,06:11:00,CHOQUE,2018-03-01,328860,19.496948,-99.073225,TIXTLA,SAN JUAN DE LOS LAGOS,SAN FELIPE DE JESUS,...,NaN,NaN,Lunes,MEDIA,NaN,ERUM,NaN,SI,Vehículo 2,AUTOMOVIL


Guardamos perfil inicial y contratos de lectura.


## 5. Delimitamos los cinco archivos y guardamos evidencias
Aplicamos ambos criterios sobre una copia de cada tabla. Registramos **motivos separados de exclusión**; un registro podría incumplir más de un criterio. Guardamos los CSV conservados y excluidos con nombres distintos, sin sobrescribir los originales. Agregamos únicamente la columna auxiliar `alcaldia_normalizada` a las salidas.

In [6]:
resumen = []
revisiones = []
with zipfile.ZipFile(ZIP, 'r') as z:
    for nombre, ruta in rutas_ssc.items():
        datos, _ = leer_csv_zip(z, ruta)
        faltantes = [c for c in ['fecha_evento', 'alcaldia'] if c not in datos.columns]
        if faltantes:
            raise KeyError(f'{nombre}: faltan columnas requeridas {faltantes}')

        fechas = pd.to_datetime(datos['fecha_evento'], format='%Y-%m-%d', errors='coerce')
        fecha_invalida = fechas.isna()
        fuera_periodo = fechas.notna() & (fechas < pd.Timestamp('2018-01-01'))
        alcaldia_norm = datos['alcaldia'].map(normalizar_alcaldia)
        fuera_alcaldia = ~alcaldia_norm.isin(ALCALDIAS)
        conservar = ~(fecha_invalida | fuera_periodo | fuera_alcaldia)

        trabajo = datos.copy()
        trabajo['alcaldia_normalizada'] = alcaldia_norm
        conservados = trabajo.loc[conservar].copy()
        excluidos = trabajo.loc[~conservar].copy()
        excluidos['motivo_fecha_invalida'] = fecha_invalida.loc[~conservar].values
        excluidos['motivo_fuera_periodo'] = fuera_periodo.loc[~conservar].values
        excluidos['motivo_alcaldia_no_valida'] = fuera_alcaldia.loc[~conservar].values

        base = Path(nombre).stem
        conservados.to_csv(DESTINO / f'{base}_etapa1.csv', index=False, encoding='utf-8-sig')
        excluidos.to_csv(DESTINO / f'{base}_excluidos_etapa1.csv', index=False, encoding='utf-8-sig')

        resumen.append({
            'archivo': nombre, 'recibidas': len(datos), 'conservadas': len(conservados),
            'excluidas': len(excluidos), 'fecha_invalida': int(fecha_invalida.sum()),
            'fuera_periodo': int(fuera_periodo.sum()),
            'fuera_o_sin_alcaldia_valida': int(fuera_alcaldia.sum()),
            'estado': 'FILTRADO'
        })
        if len(excluidos):
            columnas = [c for c in ['folio', 'fecha_evento', 'alcaldia', 'alcaldia_normalizada',
                                      'motivo_fecha_invalida', 'motivo_fuera_periodo',
                                      'motivo_alcaldia_no_valida'] if c in excluidos.columns]
            revisiones.append(excluidos[columnas].assign(archivo=nombre))
        assert len(datos) == len(conservados) + len(excluidos), f'No cuadra {nombre}'
        assert conservados['alcaldia_normalizada'].isin(ALCALDIAS).all(), f'Alcaldía no válida: {nombre}'
        print(f'{nombre}: {len(datos):,} → {len(conservados):,} conservadas; {len(excluidos):,} excluidas')

resultado_etapa1 = pd.DataFrame(resumen)
resultado_etapa1.to_csv(DESTINO / 'validaciones_etapa1_SSC.csv', index=False, encoding='utf-8-sig')
revisiones_df = pd.concat(revisiones, ignore_index=True) if revisiones else pd.DataFrame()
revisiones_df.to_csv(DESTINO / 'revision_exclusiones_SSC.csv', index=False, encoding='utf-8-sig')
display(resultado_etapa1)
print('Total filas recibidas:', int(resultado_etapa1['recibidas'].sum()))
print('Total filas conservadas:', int(resultado_etapa1['conservadas'].sum()))
print('Total filas excluidas:', int(resultado_etapa1['excluidas'].sum()))
print('NOTA: los totales suman filas de tablas diferentes, NO eventos únicos.')

HechosTransito_SSC.csv: 134,079 → 134,078 conservadas; 1 excluidas
PersonasLyFporEdad_SSC.csv: 158,077 → 158,076 conservadas; 1 excluidas
PersonasLyFporSexo_SSC.csv: 144,044 → 144,043 conservadas; 1 excluidas
PersonasLyFporTipoPersona_SSC.csv: 107,795 → 107,794 conservadas; 1 excluidas
VehiculosInvolucrados_SSC.csv: 215,079 → 215,078 conservadas; 1 excluidas


,archivo,recibidas,conservadas,excluidas,fecha_invalida,fuera_periodo,fuera_o_sin_alcaldia_valida,estado
0,HechosTransito_SSC.csv,134079,134078,1,0,0,1,FILTRADO
1,PersonasLyFporEdad_SSC.csv,158077,158076,1,0,0,1,FILTRADO
2,PersonasLyFporSexo_SSC.csv,144044,144043,1,0,0,1,FILTRADO
3,PersonasLyFporTipoPersona_SSC.csv,107795,107794,1,0,0,1,FILTRADO
4,VehiculosInvolucrados_SSC.csv,215079,215078,1,0,0,1,FILTRADO


Total filas recibidas: 759074
Total filas conservadas: 759069
Total filas excluidas: 5
NOTA: los totales suman filas de tablas diferentes, NO eventos únicos.


## 6. Verificamos archivos generados y registros excluidos
Comprobamos que existen exactamente cinco CSV conservados y cinco CSV de exclusiones. **No utilizamos `glob("*_etapa1.csv")` sin filtrar**, porque ese patrón también encuentra los archivos cuyo nombre termina en `_excluidos_etapa1.csv`.

In [7]:
conservados_csv = sorted(p for p in DESTINO.glob('*_etapa1.csv')
                        if not p.name.endswith('_excluidos_etapa1.csv'))
excluidos_csv = sorted(DESTINO.glob('*_excluidos_etapa1.csv'))
assert len(conservados_csv) == 5, f'Se esperaban 5 archivos conservados, hay {len(conservados_csv)}'
assert len(excluidos_csv) == 5, f'Se esperaban 5 archivos excluidos, hay {len(excluidos_csv)}'

for ruta in conservados_csv:
    df = pd.read_csv(ruta, dtype=str, low_memory=False)
    esperado = resultado_etapa1.loc[resultado_etapa1['archivo'].str.replace('.csv', '', regex=False) == ruta.stem.replace('_etapa1', ''), 'conservadas'].iloc[0]
    assert len(df) == esperado, f'Conteo inconsistente: {ruta.name}'
    print(f'OK {ruta.name}: {len(df):,} filas')

print('\nREGISTROS EXCLUIDOS:')
if not revisiones_df.empty:
    display(revisiones_df)
else:
    print('No se encontraron exclusiones.')
print('\nEvidencias guardadas en:', DESTINO)

OK HechosTransito_SSC_etapa1.csv: 134,078 filas
OK PersonasLyFporEdad_SSC_etapa1.csv: 158,076 filas
OK PersonasLyFporSexo_SSC_etapa1.csv: 144,043 filas
OK PersonasLyFporTipoPersona_SSC_etapa1.csv: 107,794 filas
OK VehiculosInvolucrados_SSC_etapa1.csv: 215,078 filas

REGISTROS EXCLUIDOS:


,folio,fecha_evento,alcaldia,alcaldia_normalizada,motivo_fecha_invalida,motivo_fuera_periodo,motivo_alcaldia_no_valida,archivo
0,C5/20231210/05314,2023-12-10,AV INSURGENTES,AV INSURGENTES,False,False,True,HechosTransito_SSC.csv
1,C5/20231210/05314,2023-12-10,AV INSURGENTES,AV INSURGENTES,False,False,True,PersonasLyFporEdad_SSC.csv
2,C5/20231210/05314,2023-12-10,AV INSURGENTES,AV INSURGENTES,False,False,True,PersonasLyFporSexo_SSC.csv
3,C5/20231210/05314,2023-12-10,AV INSURGENTES,AV INSURGENTES,False,False,True,PersonasLyFporTipoPersona_SSC.csv
4,C5/20231210/05314,2023-12-10,AV INSURGENTES,AV INSURGENTES,False,False,True,VehiculosInvolucrados_SSC.csv



Evidencias guardadas en: /home/ed0680e4-5639-48fc-ba69-7b1ab876af2f/Avance02_Proyecto/Proyecto Inteligencia de Negocios/resultados_SSC/etapa1


## 7. Interpretación y conclusión
Al revisar los resultados de nuestra ejecución, registramos las cantidades efectivamente obtenidas en `validaciones_etapa1_SSC.csv`. En la revisión realizada por el equipo, los cinco archivos compartieron un registro excluido con folio `C5/20231210/05314`, fecha `2023-12-10` y alcaldía `AV INSURGENTES`. Este valor nombra una vialidad y no permite asignar una alcaldía sin evidencia adicional. El folio compartido sugiere relación entre tablas, pero no prueba que sean cinco hechos distintos ni que las filas sean duplicados eliminables.

**Resultados observados en nuestra revisión previa:** 759,074 filas recibidas, 759,069 conservadas y 5 excluidas, sumando las cinco tablas. Estos resultados deben comprobarse con la salida de esta ejecución; no se utilizan como valores forzados por el código.

**Decisión:** conservamos los registros originales y las exclusiones para auditoría. No imputamos alcaldías, no borramos folios repetidos ni deduplicamos filas en esta etapa. La Etapa 2 realizará el perfilamiento y la revisión de calidad sobre los **cinco archivos conservados**, atendiendo las distintas unidades de observación.

**Archivos de evidencia producidos:** `contratos_lectura_SSC.json`, `perfil_crudo_columnas_SSC.csv`, `validaciones_etapa1_SSC.csv`, `revision_exclusiones_SSC.csv`, cinco archivos `*_etapa1.csv` y cinco `*_excluidos_etapa1.csv`.

## 8. Preparación para la Etapa 2 (sin ejecutarla aquí)
En el siguiente notebook trabajaremos con la carpeta que imprime `DESTINO`. Al seleccionar los archivos delimitados, debemos excluir expresamente los nombres terminados en `_excluidos_etapa1.csv`. De este modo, no mezclaremos los datos aptos con las evidencias de exclusión.

```python
from pathlib import Path
carpeta_ssc = Path(DESTINO)
archivos_aptos = sorted(p for p in carpeta_ssc.glob('*_etapa1.csv')
                       if not p.name.endswith('_excluidos_etapa1.csv'))
assert len(archivos_aptos) == 5
```

La Etapa 2 tendrá su propio notebook y sus propias evidencias de validación.